# DyslexAI Audio Bridge — Gemma 4 E4B

Este notebook é uma exploração paralela ao pipeline principal de imagem.

Objetivo:
- carregar um ficheiro de áudio curto (`.wav` recomendado)
- enviar o áudio para o Gemma 4 E4B
- obter uma resposta pedagógica estruturada em JSON

Saída esperada:
- `transcription`
- `clean_text`
- `words`

> Nota: o suporte multimodal de áudio depende da variante do modelo e da versão das bibliotecas disponíveis no ambiente Kaggle.

## 1) Instalação

No Kaggle, esta célula pode demorar alguns minutos na primeira execução.

In [1]:
!pip install -q -U transformers accelerate soundfile librosa ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 107.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 101.9 MB/s eta 0:00:0000:01


## 2) Imports e utilitários

In [2]:
import io
import json
import traceback

import numpy as np
import soundfile as sf
import librosa
import torch

from IPython.display import Audio, display, JSON
from ipywidgets import FileUpload, Output, VBox
from transformers import AutoProcessor, AutoModelForMultimodalLM

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 3) Configuração do modelo

Podes usar diretamente o modelo do Hugging Face ou ajustar `MODEL_ID` se estiveres a montar o modelo a partir de um caminho local/dataset do Kaggle.

In [3]:
MODEL_ID = 'google/gemma-4-e4b-it'
TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
DEVICE_MAP = 'auto'
MAX_NEW_TOKENS = 256

print('MODEL_ID =', MODEL_ID)
print('TORCH_DTYPE =', TORCH_DTYPE)

MODEL_ID = google/gemma-4-e4b-it
TORCH_DTYPE = torch.float16


## 4) Carregar processor e modelo

Se esta célula falhar, o problema pode estar na disponibilidade do modelo, autenticação, ou em diferenças de suporte multimodal na stack instalada no Kaggle.

In [4]:
processor = None
model = None

try:
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        dtype="auto",
        device_map="auto"
    )
    print('Modelo e processor carregados com sucesso.')
except Exception:
    print('Falha ao carregar modelo/processor:')
    traceback.print_exc()

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Modelo e processor carregados com sucesso.


## 5) Upload de áudio

Usa `.wav` sempre que possível. Mantém o clip curto (idealmente abaixo de 30 segundos).

In [5]:
uploader = FileUpload(accept='.wav,.mp3,.flac,.ogg,.m4a', multiple=False)
out_upload = Output()

def show_uploaded_audio(change=None):
    with out_upload:
        out_upload.clear_output()
        if not uploader.value:
            print('Ainda não foi carregado nenhum ficheiro.')
            return
        try:
            item = next(iter(uploader.value)) if isinstance(uploader.value, dict) else uploader.value[0]
            if isinstance(uploader.value, dict):
                fileinfo = uploader.value[item]
            else:
                fileinfo = item
            name = fileinfo.get('name', 'audio')
            content = fileinfo['content']
            audio_array, sr = sf.read(io.BytesIO(content))
            print(f'Ficheiro: {name}')
            print('Shape original:', np.shape(audio_array))
            print('Sample rate:', sr)
            display(Audio(audio_array, rate=sr))
        except Exception:
            traceback.print_exc()

uploader.observe(show_uploaded_audio, names='value')
display(VBox([uploader, out_upload]))

## 6) Funções auxiliares de áudio

In [6]:
def load_uploaded_audio(uploader_widget, target_sr=16000):
    if not uploader_widget.value:
        raise ValueError('Nenhum ficheiro de áudio foi carregado.')

    item = next(iter(uploader_widget.value)) if isinstance(uploader_widget.value, dict) else uploader_widget.value[0]
    fileinfo = uploader_widget.value[item] if isinstance(uploader_widget.value, dict) else item

    name = fileinfo.get('name', 'audio')
    raw = fileinfo['content']

    audio, sr = sf.read(io.BytesIO(raw))

    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    if sr != target_sr:
        audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    audio = audio.astype(np.float32)
    return name, audio, sr


import json
import re

def extract_json_block(text):
    if not text:
        return None

    # 1) tentar bloco ```json ... ```
    fenced = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fenced:
        candidate = fenced.group(1)
        try:
            return json.loads(candidate)
        except Exception:
            pass

    # 2) tentar qualquer bloco ``` ... ```
    fenced_generic = re.search(r"```\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fenced_generic:
        candidate = fenced_generic.group(1)
        try:
            return json.loads(candidate)
        except Exception:
            pass

    # 3) tentar apanhar o último objeto JSON no texto
    matches = re.findall(r"\{.*?\}", text, re.DOTALL)
    for candidate in reversed(matches):
        try:
            return json.loads(candidate)
        except Exception:
            continue

    return None

## 7) Prompt pedagógico

Aqui o objetivo não é apenas transcrever. O Gemma deve devolver uma estrutura útil para o teu leitor guiado.

In [7]:
PROMPT = """You are an educational assistant for dyslexic students.

You will receive a short audio clip spoken by a student in European Portuguese or simple Portuguese.

Tasks:
1. Transcribe exactly what the student said.
2. Rewrite the sentence with clean punctuation and capitalization.
3. Split the sentence into individual words for guided reading.
4. Keep the meaning and wording as close as possible to the spoken sentence.

Return only valid JSON in this format:
{
  \"transcription\": \"...\",
  \"clean_text\": \"...\",
  \"words\": [\"...\", \"...\"],
  \"language\": \"pt\"
}
"""

print(PROMPT)

You are an educational assistant for dyslexic students.

You will receive a short audio clip spoken by a student in European Portuguese or simple Portuguese.

Tasks:
1. Transcribe exactly what the student said.
2. Rewrite the sentence with clean punctuation and capitalization.
3. Split the sentence into individual words for guided reading.
4. Keep the meaning and wording as close as possible to the spoken sentence.

Return only valid JSON in this format:
{
  "transcription": "...",
  "clean_text": "...",
  "words": ["...", "..."],
  "language": "pt"
}



## 8) Inferência multimodal

Esta célula tenta primeiro a chamada mais direta com `audio=...` no processor. Se a stack não suportar esta assinatura, ficas logo com erro explícito para depuração.

In [8]:
def run_audio_inference(audio_array, sr, max_new_tokens=256):
    import numpy as np
    import torch

    audio_array = np.asarray(audio_array, dtype=np.float32)

    # garantir mono
    if audio_array.ndim > 1:
        audio_array = audio_array.mean(axis=1)

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "You are an educational assistant for dyslexic students.\n"
                        "Tasks:\n"
                        "1. Transcribe exactly what the student said.\n"
                        "2. Rewrite the sentence clearly.\n"
                        "3. Split the sentence into words for guided reading.\n\n"
                        "Respond only in valid JSON with this schema:\n"
                        "{\n"
                        '  "transcription": "...",\n'
                        '  "clean_text": "...",\n'
                        '  "words": ["...", "..."],\n'
                        '  "language": "..."\n'
                        "}"
                    )
                },
                {
                    "type": "audio",
                    "audio": audio_array
                }
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens)

    decoded = processor.batch_decode(
        output,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    return decoded

## 9) Executar teste com o ficheiro carregado

In [9]:
try:
    filename, audio_array, sr = load_uploaded_audio(uploader)
    print('Ficheiro preparado:', filename)
    print('Shape final:', audio_array.shape)
    print('Sample rate final:', sr)

    raw_response = run_audio_inference(audio_array, sr)
    print('--- RAW RESPONSE ---')
    print(raw_response)

    parsed = extract_json_block(raw_response)
    if parsed is not None:
        print('--- JSON PARSED ---')
        display(JSON(parsed))
    else:
        print('Não foi possível extrair JSON válido da resposta.')
except Exception:
    traceback.print_exc()

Ficheiro preparado: WhatsApp-Audio-2026-04-16-at-09.26.12.mp3
Shape final: (102818,)
Sample rate final: 16000
--- RAW RESPONSE ---
user
You are an educational assistant for dyslexic students.
Tasks:
1. Transcribe exactly what the student said.
2. Rewrite the sentence clearly.
3. Split the sentence into words for guided reading.

Respond only in valid JSON with this schema:
{
  "transcription": "...",
  "clean_text": "...",
  "words": ["...", "..."],
  "language": "..."
}
model
```json
{
  "transcription": "Percebes quando calculas dá a metros quadrados.",
  "clean_text": "Percebes que quando calculas, dá metros quadrados.",
  "words": [
    "Percebes",
    "que",
    "quando",
    "calculas",
    "dá",
    "metros",
    "quadrados"
  ],
  "language": "Portuguese"
}
```
--- JSON PARSED ---


<IPython.core.display.JSON object>

## 10) Observações para integração futura

Se este notebook funcionar, o passo seguinte é expor esta lógica atrás de um endpoint, por exemplo:

- `POST /process-audio`

Resposta sugerida do backend:

```json
{
  "transcription": "...",
  "clean_text": "...",
  "words": ["...", "..."],
  "language": "pt"
}
```

Depois o frontend pode reutilizar o mesmo leitor guiado que já existe para o fluxo de imagem.